# Workshop Databricks — Data Engineering & Data Analytics

### Da ingestão ao insight — engenharia, governança e análise em uma única plataforma

Este roteiro percorre o ciclo de vida completo de um dado de suporte de TI, focando nas capacidades de **engenharia de dados** e **análise de dados**: **governança** (Unity Catalog), **ETL** (código, declarativo e visual), **camada semântica**, **BI**, **AI Functions** e **assistência por IA** (Genie Code) — sempre sobre a mesma fundação Delta Lake + Unity Catalog.

**Público-alvo:** Engenheiros de Dados e Analistas de Dados

**Agenda:**
| Horário      | Atividade                                              |
|--------------|--------------------------------------------------------|
| 09:00–09:10  | Abertura e Contexto                                    |
| 09:10–09:25  | Unity Catalog — Governança, Linhagem e Segurança        |
| 09:25–09:45  | ETL Tradicional — Notebooks, Jobs e Arquitetura Medallion |
| 09:45–10:05  | ETL Declarativo com SDP (Lakeflow Spark Declarative Pipelines) |
| 10:05–10:20  | Lakeflow Designer — ETL Visual (No-code)               |
| 10:20–10:30  | Metric View — Camada Semântica Centralizada             |
| 10:30–10:45  | SQL Editor — Assistente AI e AI Functions               |
| 10:45–11:00  | Genie — Perguntas em Linguagem Natural                 |
| 11:00–11:15  | AI/BI Dashboard                                        |
| 11:15–11:25  | Coffee Break                                           |
| 11:25–12:00  | Genie Code — Assistente de IA no Workspace              |

**Schema:** Configurável via `00_Config` (default: `workshop_catalog.it_support`)

> ⚙️ **Antes de começar:** edite o notebook `00_Config` com o catalog e schema do ambiente, depois execute `00_Setup_Workshop` para criar as tabelas.

**Tabelas disponíveis:**
| Tabela | Descrição |
| --- | --- |
| `tb_agents` | Dados dos atendentes de suporte |
| `tb_sla` | Dados de SLA dos tickets |
| `tb_tickets` | Dados dos tickets de suporte |
| `tb_tickets_comments` | Comentários dos tickets |

**Seções da Demo:**
1. Unity Catalog — Governança e Linhagem
2. ETL Tradicional — Notebooks, Jobs e Arquitetura Medallion
3. ETL Declarativo com SDP (Lakeflow Spark Declarative Pipelines)
4. Lakeflow Designer — ETL Visual (No-code)
5. Metric View — Camada Semântica Centralizada
6. SQL Editor — Assistente e AI Functions
7. Genie — Perguntas em linguagem natural
8. Dashboard AI/BI
9. Genie Code — Assistente de IA Integrado ao Workspace

> 🧭 **Fio condutor da narrativa:** os dados brutos de tickets de TI passam por **ETL** (código, declarativo e visual) → viram **métricas** padronizadas → alimentam **dashboards** e **Genie** → e o **Genie Code** ajuda o time a construir tudo isso mais rápido, com skills de padronização e criação assistida. Uma plataforma, um dado, ponta a ponta.

## 1. Unity Catalog

### Objetivo: Mostrar governança, linhagem, versionamento e segurança dos dados

---

### 1.1 Navegação no Catálogo
- Acesse **Catalog** no menu lateral
- Navegue até `cosin_aws_serverless_catalog` > `it_support`
- Mostre as tabelas disponíveis

### 1.2 Detalhes da Tabela
- Clique em `tb_tickets`
- Mostre as abas:
  - **Columns**: schema, tipos de dados, comentários das colunas
  - **Sample Data**: preview dos dados
  - **Details**: owner, created, last modified, storage location
  - **History**: versões da tabela (Delta Time Travel)
  - **Lineage**: de onde vêm os dados e para onde vão
  - **Insights**: estatísticas de uso, queries frequentes, usuários que acessam

### 1.3 Permissões (Privileges)
- Na aba **Permissions** da tabela, mostre:
  - Como conceder `SELECT`, `MODIFY` para usuários/grupos
  - Herança de permissões do schema/catalog
  - Princípio de menor privilégio

### 1.4 Insights & Lineage
- Volte à tabela `tb_tickets`
- Mostre a aba **Lineage** para visualizar o fluxo de dados
- Mostre a aba **Insights** para:
  - Queries mais frequentes que usam a tabela
  - Usuários que mais acessam
  - Dashboards e notebooks dependentes

## 2. ETL Tradicional - Notebooks, Jobs e Arquitetura Medallion

### Objetivo: Mostrar como construir um pipeline de dados usando notebooks, jobs e a arquitetura medallion (Bronze → Silver → Gold)

---

### 2.1 Conceitos Fundamentais

#### O que é um Notebook?
- **Notebook** é um documento interativo que combina código (Python, SQL, Scala, R), visualizações e texto
- Permite execução célula por célula para desenvolvimento e depuração
- Ideal para explorar dados, criar transformações e documentar o processo
- Suporta múltiplas linguagens no mesmo notebook (`%python`, `%sql`, `%md`)

#### O que é um Job?
- **Job** é a forma de agendar e orquestrar a execução de notebooks em produção
- Pode ser executado sob demanda ou com agendamento (cron, horário, diário, etc.)
- Possui monitoramento de execução, logs, alertas de falha e retry automático
- Permite definir cluster específico ou serverless para execução

#### O que é uma Task?
- **Task** é uma unidade de trabalho dentro de um Job
- Cada Task executa um notebook, script Python, JAR ou pipeline
- Tasks podem ter **dependências entre si** formando um **DAG** (grafo acíclico dirigido)
- Exemplo: Task Silver depende de Task Bronze; Task Gold depende de Task Silver

#### Arquitetura Medallion (Bronze → Silver → Gold)
| Camada | Propósito | Exemplo |
| --- | --- | --- |
| **Bronze** | Dados brutos, ingestão sem transformação | `tb_tickets`, `tb_agents`, `tb_sla`, `tb_tickets_comments` |
| **Silver** | Dados limpos, enriquecidos com joins e campos calculados | `silver_tickets_full`, `silver_comments_enriched` |
| **Gold** | Dados agregados, prontos para dashboards e relatórios | `gold_support_kpis`, `gold_tickets_by_country`, `gold_agent_performance`, `gold_sla_compliance` |

---

### 2.2 Notebooks de ETL Criados

#### Notebook Silver
📓 ETL Silver - IT Support (na pasta `ETL Tradicional/`)
- Lê as 4 tabelas fonte (Bronze)
- Cria `silver_tickets_full`: join de tickets + agents + sla com campos calculados:
  - `resolution_hours` — tempo de resolução em horas
  - `first_response_hours` — tempo até primeira resposta
  - `sla_resolution_met` — flag se resolveu dentro do SLA
  - `sla_first_response_met` — flag se respondeu dentro do SLA
- Cria `silver_comments_enriched`: comments com contexto do ticket (país, prioridade, tópico)

#### Notebook Gold
📓 ETL Gold - IT Support (na pasta `ETL Tradicional/`)
- Lê as tabelas Silver
- Cria 4 tabelas Gold:
  - `gold_support_kpis` — KPIs consolidados (total tickets, tempo médio, % SLA, satisfação)
  - `gold_tickets_by_country` — métricas por país
  - `gold_agent_performance` — performance individual de cada atendente
  - `gold_sla_compliance` — conformidade de SLA por prioridade e país

---

### 2.3 Demonstração do ETL

**Passo 1: Abra o notebook Silver**
- Mostre a estrutura: configuração → leitura → transformação → escrita → validação
- Execute as células e mostre os resultados intermediários
- Destaque os campos calculados (resolution_hours, SLA flags)

**Passo 2: Abra o notebook Gold**
- Mostre as agregações por país, agente, prioridade
- Execute e mostre os DataFrames resultantes
- Destaque como os dados estão prontos para consumo em dashboards

**Passo 3: Volte ao Unity Catalog**
- Mostre as novas tabelas silver_* e gold_* criadas no schema
- Mostre a **Lineage** — agora as tabelas têm fluxo Bronze → Silver → Gold visível

---

### 2.4 Job de Orquestração

**DAG de Tasks:**
```
etl_silver → etl_gold
```

| Task | Notebook | Dependência |
| --- | --- | --- |
| `etl_silver` | ETL Silver - IT Support | — (executa primeiro) |
| `etl_gold` | ETL Gold - IT Support | `etl_silver` (só executa após Silver concluir) |

**O que demonstrar no Job:**
1. Abra o job e mostre o **DAG visual** das tasks (etl_silver → etl_gold)
2. Clique em **Run Now** para executar manualmente
3. Acompanhe a execução em tempo real
4. Mostre os **logs de execução** de cada task
5. Mostre o **histórico de runs** e duração
6. Mostre a configuração de **alertas** e **agendamento**

In [0]:
# =================================================================
# Insere um comentário aleatório para um ticket sem comentário,
# para demonstrar o processamento INCREMENTAL do SDP.
# Execute esta célula ANTES de re-rodar o pipeline SDP e observe
# que apenas o novo registro é processado (STREAMING TABLE).
# =================================================================

# MAGIC %run ./00_Config
spark.sql(f"""
INSERT INTO {CATALOG}.{SCHEMA}.tb_tickets_comments
SELECT
    t.ticket_id,
    CASE (abs(hash(t.ticket_id)) % 5)
        WHEN 0 THEN 'Cliente relatou que o problema persiste após reinicialização do sistema.'
        WHEN 1 THEN 'Atendente solicitou logs adicionais para diagnóstico.'
        WHEN 2 THEN 'Problema resolvido após atualização de firmware do equipamento.'
        WHEN 3 THEN 'Usuário confirmou que a solução alternativa está funcionando.'
        ELSE 'Encaminhado para equipe de infraestrutura para análise aprofundada.'
    END AS comments
FROM {CATALOG}.{SCHEMA}.tb_tickets t
LEFT JOIN {CATALOG}.{SCHEMA}.tb_tickets_comments c
    ON t.ticket_id = c.ticket_id
WHERE c.ticket_id IS NULL
LIMIT 1
""")
print("✅ Comentário inserido para teste incremental")

## 3. ETL Declarativo com Lakeflow Spark Declarative Pipelines (SDP)

### Objetivo: Mostrar a abordagem **moderna e declarativa** para ETL — o mesmo resultado da Seção 2, mas com pipeline gerenciado, qualidade de dados nativa e zero orquestração manual

> 🔗 **Ligação com a Seção 2:** enquanto na seção anterior construímos o ETL com notebooks + jobs + DAG manual, aqui usamos **SDP** — você declara **o que quer** (tabelas e transformações) e a plataforma cuida do **como** (orquestração, dependências, retry, incremental).

---

### 3.1 O que é o SDP (Spark Declarative Pipelines)?

- **SDP** é o motor de pipelines declarativos do Databricks — você define tabelas e transformações em SQL ou Python, e a plataforma gerencia **automaticamente** a execução, dependências, retry e processamento incremental.
- Em vez de "execute o notebook A, depois o B", você declara: "a tabela Silver depende da Bronze" e o SDP **resolve o DAG sozinho**.
- Integra nativamente com Unity Catalog (governança + lineage) e roda em Serverless.

#### ETL Tradicional vs. SDP — Comparação Direta
| Aspecto | ETL Tradicional (Seção 2) | SDP (esta seção) |
| --- | --- | --- |
| **Autoria** | `CREATE OR REPLACE TABLE` em notebooks separados | `CREATE STREAMING TABLE` / `CREATE MATERIALIZED VIEW` em um único notebook |
| **Orquestração** | Job com DAG manual (Task Silver → Task Gold) | **Automática** — SDP resolve dependências |
| **Processamento** | Full rewrite a cada execução | **Incremental** (STREAMING TABLE) ou **auto-refresh** (MATERIALIZED VIEW) |
| **Qualidade de dados** | Validação manual (IF/CASE no código) | **Expectations** nativas com 3 modos (DROP ROW, FAIL UPDATE, WARN) |
| **Monitoramento** | Logs do Job + alertas manuais | **Dashboard de qualidade** integrado no pipeline |
| **Retry/Recovery** | Configuração manual no Job | **Automático** — SDP sabe de onde retomar |
| **Notebooks** | 2 notebooks + 1 Job | **1 notebook** = pipeline inteiro |

---

### 3.2 Principais Vantagens do SDP

* **Processamento Incremental Automático**: Com `STREAMING TABLE`, o SDP processa apenas dados novos — sem varredura, sem re-escrita
* **Qualidade de Dados Nativa (Expectations)**: Declare regras uma vez — DROP ROW (limpa), FAIL UPDATE (bloqueia), WARN (monitora)
* **DAG Automático**: Um único notebook, a plataforma monta o DAG visual automaticamente
* **Recovery Inteligente**: Se falha na Gold, retoma de onde parou — não reprocessa a Silver
* **Observabilidade Integrada**: Métricas de qualidade em tempo real por tabela e regra
* **Custo Otimizado**: Processa menos dados = menos compute = menor custo

#### STREAMING TABLE vs. MATERIALIZED VIEW
| Tipo | Quando usar | Comportamento |
| --- | --- | --- |
| **STREAMING TABLE** | Dados que crescem (append) — ex: novos tickets | Processa **apenas dados novos** a cada refresh |
| **MATERIALIZED VIEW** | Agregações e transformações complexas — ex: KPIs | **Recalcula tudo** a cada refresh |

#### Expectations (Regras de Qualidade)
| Modo | Comportamento | Quando usar |
| --- | --- | --- |
| **DROP ROW** | Remove linhas inválidas silenciosamente | Limpeza na Silver |
| **FAIL UPDATE** | **Para o pipeline** se houver violação | Dados críticos que não podem ter erro |
| **WARN** | Apenas registra métrica no dashboard | Monitoramento sem bloqueio |

---

### 3.3 O Notebook SDP Criado

📓 SDP ETL - IT Support (Silver e Gold) — na subpasta `SDP Pipeline` deste workshop

Um **único notebook** com todo o pipeline:

```
┌─────────────────────┐     ┌─────────────────────┐
│  tb_tickets         │     │  tb_tickets_comments │
│  tb_agents          │     │  (Bronze)            │
│  tb_sla (Bronze)    │     │                     │
└────────┬────────────┘     └────────┬────────────┘
         │                           │
         ▼                           ▼
┌──────────────────────────┐ ┌──────────────────────────────┐
│ sdp_silver_tickets_full  │ │ sdp_silver_comments_enriched │
│ STREAMING TABLE          │ │ STREAMING TABLE              │
└────────┬─────────────────┘ └──────────────────────────────┘
         │
         ▼
┌───────────────────────────────────────────────────────┐
│              MATERIALIZED VIEWS (Gold)                │
│  sdp_gold_support_kpis      (KPIs consolidados)       │
│  sdp_gold_tickets_by_country (métricas por país)       │
│  sdp_gold_agent_performance  (por atendente)           │
│  sdp_gold_sla_compliance     (SLA crítico/FAIL UPDATE) │
└───────────────────────────────────────────────────────┘
```

---

### 3.4 Como Criar e Executar o Pipeline

**Passo 1: Crie o Pipeline no Databricks**
1. Menu lateral → **+ New** → **ETL Pipelines**
2. Configure:
   - **Nome:** `SDP - IT Support (Silver e Gold)`
   - **... Add Source Code:** selecione o notebook `SDP ETL - IT Support (Silver e Gold)`
   - **Target catalog:** `<seu_catalog>` (o mesmo do `00_Config`)
   - **Target schema:** `<seu_schema>` (o mesmo do `00_Config`)
   - **Compute:** Serverless
3. Em **Settings → Configuration** (Advanced), adicione:
   - Chave: `source_schema` → Valor: `<seu_catalog>.<seu_schema>` (ex: `workshop_catalog.it_support`)

**Passo 2: Execute o Pipeline**
1. Clique em **Start** para executar
2. Acompanhe o **DAG visual**
3. Observe as **métricas de qualidade** em cada nó

**Passo 3: Explore o Dashboard de Qualidade**
- Clique em qualquer nó do DAG e veja a aba **Data Quality**

---

### 3.5 Comparativo Final

<table>
<tr><th>Critério</th><th>Seção 2 (Tradicional)</th><th>Seção 3 (SDP)</th></tr>
<tr><td><b>Arquivos</b></td><td>2 notebooks + 1 Job</td><td>1 notebook + 1 Pipeline</td></tr>
<tr><td><b>Dependências</b></td><td>DAG manual no Job</td><td>Automáticas</td></tr>
<tr><td><b>Processamento</b></td><td>Full overwrite</td><td>Incremental</td></tr>
<tr><td><b>Qualidade</b></td><td>Sem validação nativa</td><td>Expectations (DROP/FAIL/WARN)</td></tr>
<tr><td><b>Monitoramento</b></td><td>Logs do Job</td><td>Dashboard de qualidade integrado</td></tr>
<tr><td><b>Recovery</b></td><td>Retry manual</td><td>Checkpoint automático</td></tr>
</table>

> 🎯 **Mensagem:** o SDP não substitui o ETL em notebook — **complementa**. Engenheiros que preferem controle total usam notebooks + Jobs. Equipes que querem **produtividade e qualidade built-in** usam SDP. Ambos rodam na mesma plataforma.

## 4. Lakeflow Designer — ETL Visual (No-code)

### Objetivo: Construir um pipeline Silver → Gold **conversando em português** com o Designer, sem escrever código

---

### 4.1 O que é o Lakeflow Designer?

- O **Lakeflow Designer** (também chamado *Visual Data Prep*) é um editor **visual** para construir pipelines de ETL **sem código** — e conta com um **assistente de IA (Genie Code)** que gera as transformações a partir de **instruções em linguagem natural**.
- Cada fluxo gera, por baixo dos panos, um **pipeline Lakeflow declarativo real** — o mesmo artefato que um engenheiro escreveria em SQL/PySpark.
- Herda tudo da plataforma: **Unity Catalog** (governança + lineage), **Serverless** e agendamento como **Job**.

#### Designer vs. ETL em código — quando usar cada um?
| Aspecto | ETL em Notebook/Código (Seção 2) | Lakeflow Designer |
| --- | --- | --- |
| Público | Engenheiro de dados | Analista de dados / negócio |
| Autoria | SQL / PySpark | Visual + linguagem natural (Genie Code) |
| Resultado | Pipeline Lakeflow | **Pipeline Lakeflow (mesmo motor!)** |
| Governança | Unity Catalog | Unity Catalog |

> 💡 **Não é uma ferramenta paralela** — é a **mesma engine** de pipeline, com uma porta de entrada visual e assistida por IA.

---

### 4.2 Como usar (passo a passo)

1. Menu lateral → **Data Engineering** → **Visual Data Prep**
2. Selecione o catalog `cosin_aws_serverless_catalog` e o schema `it_support`
3. Abra o **assistente (Genie Code)** e **cole os prompts abaixo, um de cada vez**
4. A cada passo, veja o **preview dos dados** e o nó criado no fluxo visual
5. Ao final, clique em **Run** e mostre o **DAG** e a **lineage** no Unity Catalog

---

### 4.3 🥈 Prompt 1 — Camada Silver (colar no Designer)

> Crie uma tabela Silver chamada `designer_silver_tickets` a partir da tabela `tb_tickets`.
> Junte com a tabela `tb_agents` pela coluna `ticket_id` para trazer `agent_name` e `agent_group`.
> Junte com a tabela `tb_sla` pela coluna `ticket_id` para trazer `survey_results`, `resolution_time` e `expected_sla_to_resolve`.
> Crie uma coluna nova `tempo_resolucao_horas` com a diferença em horas entre `close_time` e `created_time`.
> Crie uma coluna nova `dentro_sla` que seja verdadeira quando `close_time` for menor ou igual a `expected_sla_to_resolve`.
> Padronize a coluna `priority` deixando a primeira letra maiúscula.
> Mantenha apenas as linhas em que `country` não é nulo.

---

### 4.4 🥇 Prompt 2 — Camada Gold (colar no Designer)

> Crie uma tabela Gold chamada `designer_gold_kpis_por_pais` a partir de `designer_silver_tickets`.
> Agrupe por `country` e calcule: a quantidade total de tickets, o tempo médio de resolução em horas,
> a nota média de satisfação (`survey_results`) e o percentual de tickets dentro do SLA (`dentro_sla`).
> Ordene do país com mais tickets para o com menos.

---

### 4.5 Validação (rodar no SQL Editor após o pipeline concluir)

```sql
SELECT * FROM cosin_aws_serverless_catalog.it_support.designer_silver_tickets LIMIT 10;
SELECT * FROM cosin_aws_serverless_catalog.it_support.designer_gold_kpis_por_pais ORDER BY 2 DESC;
```

> 🎯 **Mensagem:** o Databricks atende **os dois perfis** — o engenheiro que escreve código (Seção 2) e o analista que descreve o que quer **em português** (esta seção) — **na mesma plataforma, com a mesma governança e o mesmo motor de execução**.

## 5. Metric View — Camada Semântica Centralizada

### Objetivo: Mostrar como centralizar definições de métricas de negócio em um único lugar

---

### 5.1 O que é uma Metric View?

- **Metric View** é um objeto do Unity Catalog que centraliza definições de métricas de negócio
- Diferente de views tradicionais, separa **dimensões** (por onde agrupar) de **medidas** (o que calcular)
- Permite que dashboards, Genie e notebooks usem as **mesmas definições** de KPIs
- Elimina o problema de "cada dashboard calcula diferente"

#### Componentes de uma Metric View
| Componente | O que é | Exemplo |
| --- | --- | --- |
| **Source** | Tabela ou query base | `silver_tickets_full` |
| **Dimensions** | Colunas categóricas para agrupar | País, Prioridade, Atendente |
| **Measures** | Cálculos agregados (KPIs) | Total de Tickets, SLA %, Satisfação |
| **Metadata** | Comentários, sinônimos, formatação | Ajuda a IA a entender as métricas |

---

### 5.2 VIEW vs METRIC VIEW — Na prática

Cada pergunta de negócio exige uma **VIEW específica** com agregação fixa. Com uma **única Metric View**, todas são respondidas mudando apenas o `SELECT` e `GROUP BY`.

| Abordagem | Objetos criados | Manutenção |
| --- | --- | --- |
| Views tradicionais | **4+ views** (uma por pergunta) | Cada alteração na fórmula exige atualizar **todas** as views |
| Metric View | **1 metric view** | Alteração na fórmula é feita **uma única vez** e reflete em todos os consumidores |

---

### 5.3 Metric View Criada

📊 **`cosin_aws_serverless_catalog.it_support.mv_support_metrics`**

**Dimensões (9):** País, Prioridade, Status, Tópico, Grupo de Produto, Nível de Suporte, Grupo de Atendimento, Atendente, Mês de Abertura

**Medidas (9):** Total de Tickets, Tempo Médio de Resolução (h), Tempo Médio de Primeira Resposta (h), Nota Média de Satisfação, Taxa de SLA Resolução (%), Taxa de SLA Primeira Resposta (%), Tickets Abertos, Tickets Fechados, Média de Interações

---

### 5.4 Como Consultar uma Metric View

**Regras importantes:**
- Medidas **sempre** envolvidas por `MEASURE()`
- **Sempre** usar `GROUP BY ALL`
- **Nunca** usar `SELECT *`
- Nomes com espaços em **backticks** (`` ` ``)

**Exemplos:**
```sql
-- Tickets por país
SELECT `País`, MEASURE(`Total de Tickets`) AS total
FROM mv_support_metrics
GROUP BY ALL;

-- SLA % por prioridade
SELECT `Prioridade`, MEASURE(`Taxa de SLA Resolução (%)`) AS sla_pct
FROM mv_support_metrics
GROUP BY ALL;

-- CSAT por grupo
SELECT `Grupo de Atendimento`, MEASURE(`Nota Média de Satisfação`) AS csat
FROM mv_support_metrics
GROUP BY ALL;
```

---

### 5.5 Criando a Metric View com Genie Code

**Abra o Genie Code (sidebar ou no notebook) e cole o prompt abaixo:**

> Crie uma Metric View chamada `mv_support_metrics` no schema `cosin_aws_serverless_catalog.it_support`.
>
> Cruze as tabelas `silver_tickets_full`, `tb_agents` e `tb_sla` pelo `ticket_id`.
>
> Crie as dimensões que fizerem sentido para análise de suporte de TI (país, prioridade, status, atendente, grupo, etc.) e as medidas mais relevantes para KPIs de suporte (volume de tickets, tempos médios, taxas de SLA, satisfação).
>
> Use nomes em português para dimensões e medidas, adicione comentários e sinônimos para facilitar o uso por IA e dashboards.

> 💡 **Ponto da demo:** o Genie Code entende o contexto das tabelas no Unity Catalog e **infere automaticamente** quais colunas viram dimensões, quais viram medidas, os cálculos corretos (médias, percentuais, contagens filtradas) e a formatação adequada — tudo a partir de uma instrução simples e de alto nível.

---

### 5.6 Demonstração

1. Execute o prompt acima no **Genie Code** — ele criará a Metric View automaticamente
2. Abra o **Catalog** e navegue até `cosin_aws_serverless_catalog.it_support.mv_support_metrics`
3. Mostre que é do tipo **Metric View**
4. Mostre as abas de **Dimensions** e **Measures** com metadados semânticos
5. No **SQL Editor**, execute os exemplos da seção 5.4 mostrando a flexibilidade
6. Mostre como a metric view pode ser usada diretamente em **Dashboards** e **Genie**

## 6. SQL Editor

### Objetivo: Mostrar o assistente AI e as AI Functions

📚 **Documentação:** [AI Functions Reference](https://docs.databricks.com/aws/en/large-language-models/ai-functions)

---

### 6.1 Consultas Básicas

```sql
SELECT * FROM cosin_aws_serverless_catalog.it_support.tb_agents;
SELECT * FROM cosin_aws_serverless_catalog.it_support.tb_sla;
SELECT * FROM cosin_aws_serverless_catalog.it_support.tb_tickets;
SELECT * FROM cosin_aws_serverless_catalog.it_support.tb_tickets_comments;
```

---

### 6.2 Assistente AI no SQL Editor
- Abra o SQL Editor
- Use o assistente para gerar queries pedindo em linguagem natural:
  - "Faça a análise de sentimentos da coluna comments da tabela tb_tickets_comments"
  - "Faça o join entre tb_tickets_comments e tb_sla, filtrando survey_results < 3"
  - "Agregue todos os comentários em uma única linha"

---

### 6.3 AI Functions - Análise de Sentimento

```sql
SELECT 
  comments,
  ai_analyze_sentiment(comments) AS sentimento
FROM cosin_aws_serverless_catalog.it_support.tb_tickets_comments;
```

---

### 6.4 AI Functions - Classificação

```sql
SELECT 
  comments,
  ai_classify(comments, ARRAY('hardware', 'software', 'rede', 'acesso', 'outro')) AS categoria
FROM cosin_aws_serverless_catalog.it_support.tb_tickets_comments
LIMIT 20;
```

---

### 6.5 AI Functions - Sugestões de Melhoria com ai_query

```sql
SELECT
  *,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'Você é um especialista em suporte de TI e precisa trazer 5 sugestões para melhoria de atendimento, 
    trazer o peso de importância de cada sugestão, se limite a 1000 caracteres  
    e se baseie nos seguintes comentários de atendimentos ruins: ' || all_comments
  ) AS sugestoes
FROM (
  SELECT STRING_AGG(tc.comments, ' ') AS all_comments
  FROM cosin_aws_serverless_catalog.it_support.tb_tickets_comments tc
  JOIN cosin_aws_serverless_catalog.it_support.tb_sla sc ON tc.ticket_id = sc.ticket_id
  WHERE sc.survey_results < 3
);
```

## 7. Genie

### Objetivo: Consultas em linguagem natural sobre os dados de suporte

---

### 7.1 Configuração da Sala Genie
- Acesse **Genie** no menu lateral
- Crie uma nova sala ou acesse a existente
- Adicione as tabelas:
  - `cosin_aws_serverless_catalog.it_support.tb_tickets`
  - `cosin_aws_serverless_catalog.it_support.tb_sla`
  - `cosin_aws_serverless_catalog.it_support.tb_agents`
  - `cosin_aws_serverless_catalog.it_support.tb_tickets_comments`

### 7.2 Instruções da Sala
- Me responda sempre em português
- Quando perguntarem sobre tempo de atendimento, ao fazer o cálculo de encerramento - abertura, formate em X Dia(s), Y Hora(s), Z Minuto(s)
- Se perguntarem sobre o pior atendente, ou atendente mais lento, responda que não pode fazer tal análise

### 7.3 Demonstrando AI Functions no Genie
```sql
SELECT ai_analyze_sentiment(comments) FROM cosin_aws_serverless_catalog.it_support.tb_tickets_comments
```

### 7.4 Perguntas para Demonstração
1. **Qual o pior atendente?** *(deve recusar conforme instrução)*
2. **Analise os sentimentos dos comentários, me trazendo as quantidades de cada sentimento por país**
3. **Quais foram os top 10 chamados mais demorados e seus respectivos atendentes?**
4. **Me adicione também a prioridade, nível de suporte, de cada ticket e o grupo de atendimento do atendente?**
5. **Calcule o percentual de tickets dentro do SLA por país e gere um gráfico de pizza para exibição**

### 7.5 Genie Research Agent

Pergunta para o Research Agent:
> "Analise os tickets de suporte com survey_results abaixo de 3 e identifique os principais padrões de problemas. Quais são as categorias de problemas mais comuns, os tempos médios de resolução por categoria, e quais atendentes ou grupos de atendimento têm maior incidência de avaliações negativas? Forneça insights acionáveis para melhorar a qualidade do atendimento."

## 8. AI/BI Dashboard

### Objetivo: Criar e editar dashboards com assistente AI

---

### 8.1 Editando Dashboard Existente
- Acesse o dashboard existente
- Use o assistente para adicionar/modificar visualizações:
  - Traga a quantidade de chamados por país
  - Altere para gráfico de pizza

### 8.2 Perguntas para o Assistente do Dashboard
1. **Analise os sentimentos dos comentários, me trazendo as quantidades de cada sentimento por país**
2. **Quais foram os top 10 chamados mais demorados e seus respectivos atendentes?**
3. **Me adicione também a prioridade, nível de suporte, de cada ticket e o grupo de atendimento do atendente?**
4. **Calcule o percentual de tickets dentro do SLA por país e gere um gráfico de pizza para exibição**

---

### 8.3 Criando um Dashboard do Zero com o Assistente

**Passo 1: Acesse Dashboards e clique em "Create Dashboard"**

**Passo 2: Ative o Assistente AI/BI**
- Clique no ícone do assistente no canto superior direito

**Passo 3: Descreva o que você precisa**

Exemplos de prompts:
- "Crie um dashboard de análise de suporte de TI usando as tabelas `cosin_aws_serverless_catalog.it_support.tb_tickets`, `cosin_aws_serverless_catalog.it_support.tb_sla` e `cosin_aws_serverless_catalog.it_support.tb_agents`"
- "Adicione visualizações para: quantidade de tickets por país, tempo médio de resolução, percentual de SLA atendido, e top 5 atendentes por volume de tickets"
- "Inclua filtros para período de data, país e prioridade"

**Passo 4: Refine as visualizações**
- "Altere o gráfico de barras para pizza"
- "Adicione uma tabela com os detalhes dos tickets em aberto"
- "Formate os números para mostrar percentuais com 2 casas decimais"

## 9. Genie Code — Assistente de IA Integrado ao Workspace

### Objetivo: Demonstrar o assistente de IA nativo do Databricks para geração de código, análise de dados, criação de assets e automação de tarefas

---

### 9.1 O que é o Genie Code?

O **Genie Code** é o assistente de inteligência artificial integrado nativamente ao Databricks. Ele está disponível em qualquer ponto do workspace e entende o contexto do seu ambiente — catálogos, schemas, tabelas, notebooks e dashboards — para fornecer ajuda precisa e contextualizada.

#### Onde está disponível?
| Local | O que faz |
| --- | --- |
| **Notebooks** | Gera, edita e explica células de código Python, SQL e Scala |
| **SQL Editor** | Autocompleta queries, sugere otimizações, gera SQL por linguagem natural |
| **Dashboards AI/BI** | Cria e modifica widgets via linguagem natural |
| **Sidebar global** | Chat conversacional disponível em qualquer tela do workspace |

---

### 9.2 Principais Capacidades

#### Geração de Código e SQL
- Transforma descrições em linguagem natural em código executável (Python, SQL, Spark)
- Sugere queries otimizadas com base no schema das tabelas do Unity Catalog
- Completa código automaticamente enquanto você digita

#### Explicação e Documentação
- Explica o que qualquer trecho de código faz, em linguagem natural
- Documenta funções, queries e notebooks automaticamente
- Traduz erros técnicos em explicações compreensíveis

#### Diagnóstico e Correção de Erros
- Identifica a causa de erros de execução (Python, SQL, Spark)
- Sugere e aplica correções diretamente na célula com problema
- Mantém o contexto da conversa para iterar na solução

#### Criação e Edição de Assets
- Cria notebooks completos com múltiplas células a partir de uma instrução
- Edita dashboards, queries salvas e arquivos diretamente via chat
- Renomeia, reorganiza e documenta assets do workspace

---

### 9.3 Demonstração no Notebook — Genie Code para ETL

**Acesse um Notebook no workspace e use o painel lateral do Genie Code**

#### Exemplo 1 — Geração de ETL assistido:
> "Crie uma análise exploratória dos tickets de suporte usando a tabela `cosin_aws_serverless_catalog.it_support.tb_tickets`. Inclua distribuição por prioridade, tempo médio de resolução e tendência mensal."

#### Exemplo 2 — Criação de ETL Silver com Genie Code:
> "Crie o ETL Silver: faça o join entre tb_tickets, tb_agents e tb_sla pela coluna ticket_id. Calcule o tempo de resolução em horas e crie flags de SLA. Salve como tabela silver_tickets_genie no schema cosin_aws_serverless_catalog.it_support."

#### Exemplo 3 — Diagnóstico de erro:
> "Esse código está retornando um erro de AnalysisException, me ajude a corrigir"

#### Exemplo 4 — Análise orientada a negócio:
> "Analise os dados da tabela tb_sla e me diga quais países têm mais tickets fora do SLA"

---

### 9.4 Genie Code para Criação de Salas Genie

**Use o Genie Code para criar e configurar Salas Genie automaticamente**

#### Prompt para criar uma Sala Genie:
> "Crie uma Sala Genie chamada 'IT Support — Análise de Tickets' com as tabelas:
> - cosin_aws_serverless_catalog.it_support.tb_tickets
> - cosin_aws_serverless_catalog.it_support.tb_sla
> - cosin_aws_serverless_catalog.it_support.tb_agents
> - cosin_aws_serverless_catalog.it_support.tb_tickets_comments
>
> Instruções: responder sempre em português, formatar tempos em Dia(s)/Hora(s)/Minuto(s), recusar perguntas sobre 'pior atendente'."

#### O que o Genie Code faz:
- Cria a sala com as tabelas informadas
- Configura as instruções (general instructions) da sala
- A sala fica pronta para uso imediato

---

### 9.5 Genie Code com Skills de Padronização

**Skills** são arquivos de instruções que estendem o Genie Code com conhecimento específico do seu projeto ou empresa.

#### O que são Skills?
- Arquivos `.md` que você salva no workspace (ex: `.assistant/skills/minha-skill/SKILL.md`)
- Contêm regras, padrões, nomenclaturas e boas práticas do time
- O Genie Code **lê automaticamente** e aplica ao gerar código

#### Exemplo de Skill — Padronização de ETL:
```markdown
# Skill: Padrões de ETL - IT Support

## Regras de Nomenclatura
- Tabelas Bronze: `tb_<entidade>` (ex: tb_tickets)
- Tabelas Silver: `silver_<entidade>_<sufixo>` (ex: silver_tickets_full)
- Tabelas Gold: `gold_<domínio>_<métrica>` (ex: gold_support_kpis)

## Padrões de Código
- Sempre usar INITCAP() para padronizar colunas categóricas (priority, status, source)
- Sempre incluir `etl_timestamp` com CURRENT_TIMESTAMP()
- Sempre validar NOT NULL em ticket_id e country
- Usar LEFT JOIN para não perder registros

## Catalog/Schema
- Catalog: cosin_aws_serverless_catalog
- Schema: it_support
```

#### Como demonstrar:
1. Crie o arquivo de skill no workspace
2. Peça ao Genie Code para gerar um ETL — observe que ele segue os padrões da skill
3. Modifique a skill e veja o comportamento mudar

---

### 9.6 Genie Code para Criação de Dashboards

**Use o Genie Code para criar e editar AI/BI Dashboards**

#### Prompt para criar um Dashboard via Genie Code:
> "Crie um AI/BI Dashboard chamado 'IT Support — Visão Geral' usando as tabelas do schema cosin_aws_serverless_catalog.it_support. Inclua:
> - KPIs: total de tickets, tempo médio de resolução, % SLA atendido, nota média CSAT
> - Gráfico de barras com tickets por país
> - Gráfico de pizza com distribuição por prioridade
> - Tabela com top 10 atendentes por volume
> - Filtros por país e prioridade"

#### O que o Genie Code faz:
- Cria o dashboard com os datasets necessários
- Adiciona os widgets (KPIs, gráficos, tabelas)
- Configura filtros e interações
- Dashboard fica pronto para publicação

---

### 9.7 Genie Code para Migração de Dashboards

**Converta dashboards existentes de Power BI, Tableau ou Looker para AI/BI Dashboards**

#### Formatos suportados:
| Ferramenta | Formatos |
| --- | --- |
| **Power BI** | `.pbix`, `.pbit` |
| **Tableau** | `.twb`, `.twbx`, `.tds` |
| **Looker** | `.lkml`, JSON exports |

#### Exemplo de Prompt:
> *(Faça upload do arquivo `.pbix` ou `.twb` no chat)*
> "Migre esse dashboard do Power BI para um AI/BI Dashboard do Databricks. As tabelas equivalentes estão no schema `cosin_aws_serverless_catalog.it_support`."

---

### 9.8 Genie Code para Migração de ETLs

**Converta pipelines ETL legados para PySpark ou Lakeflow**

#### Ferramentas suportadas:
| Ferramenta | Formato |
| --- | --- |
| **Alteryx** | `.yxmd`, `.yxwz` (XML) |
| **Informatica** | XML exports |
| **SSIS** | `.dtsx` packages |
| **Pentaho** | `.ktr`, `.kjb` |

#### Exemplo de Prompt:
> *(Faça upload do arquivo `.yxmd` no chat)*
> "Converta esse workflow do Alteryx para PySpark. Os dados de entrada estão em `cosin_aws_serverless_catalog.it_support` e a saída deve ser salva no mesmo schema."

---

### 9.9 Pontos para Destacar

- **Contexto nativo**: Conhece os schemas, tabelas e estruturas do seu catalog automaticamente
- **Multi-turno**: Mantém o histórico da conversa para iterações
- **Segurança e governança**: Respeita as permissões do usuário no Unity Catalog
- **Capacidade agêntica**: Executa ações, cria células, edita dashboards e gerencia assets
- **Skills de padronização**: Estende o assistente com regras do time/empresa
- **Sem lock-in de modelo**: Construído sobre a plataforma agnóstica do Databricks